# 02 - Conditional Diffusion Model (DDPM)
**Forward process (q):** gradually add Gaussian noise following a beta schedule
$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\quad \epsilon\sim\mathcal N(0,I),\quad \bar\alpha_t=\prod_{s\le t}(1-\beta_s)$$
**Training objective:** a class-conditional U-Net $\epsilon_\theta(x_t, t, y)$ predicts the added noise; loss = MSE($\epsilon_\theta$, $\epsilon$).
**Reverse process (sampling):** start from $x_T\sim\mathcal N(0,I)$ and iteratively denoise from $t=T-1$ to $0$.

In [ ]:
import os, sys, random
import numpy as np
import torch
import matplotlib.pyplot as plt

# Set QUICK_RUN=1 in the environment for a tiny smoke-test run (few samples / 1 epoch). Leave unset for the real run.
QUICK_RUN = os.environ.get("QUICK_RUN", "0") == "1"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| QUICK_RUN:", QUICK_RUN)
os.makedirs("checkpoints", exist_ok=True); os.makedirs("outputs", exist_ok=True)

In [ ]:
from data_utils.dataloader import get_dataloader
from model import ConditionalUNet
from training import (linear_beta_schedule, cosine_beta_schedule, get_beta_schedule, train_diffusion,
                      sample_images)
from utils.visualize import show_images, show_class_grid, plot_losses

# ---------------- configuration ----------------
T = 100 if QUICK_RUN else 1000          # diffusion steps used in training
BATCH_SIZE = 128
EPOCHS = 1 if QUICK_RUN else 30
LR = 2e-4
BASE_CH = 16 if QUICK_RUN else 64
SAMPLE_STEPS = 20 if QUICK_RUN else 250 # re-spaced steps used for fast sampling in the notebook
SUBSET = 512 if QUICK_RUN else None
TRAIN_COSINE = True                     # stand-out: also train a cosine-schedule model for comparison

train_loader = get_dataloader(batch_size=BATCH_SIZE, train=True, subset=SUBSET)
print("batches per epoch:", len(train_loader))

### Noise schedules (linear vs cosine)

In [ ]:
betas_linear = linear_beta_schedule(T)
betas_cosine = cosine_beta_schedule(T)
abar = lambda b: torch.cumprod(1 - b, 0)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(betas_linear, label="linear"); ax[0].plot(betas_cosine, label="cosine"); ax[0].set_title("beta_t"); ax[0].legend()
ax[1].plot(abar(betas_linear), label="linear"); ax[1].plot(abar(betas_cosine), label="cosine"); ax[1].set_title("alpha_bar_t (signal remaining)"); ax[1].legend()
plt.tight_layout(); plt.savefig("outputs/02_schedules.png", dpi=150); plt.show()

### Forward diffusion (q-sampling) demo
Implemented explicitly here from the schedule: $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$.

In [ ]:
x0, y0 = next(iter(train_loader))
x0 = x0[:8]
alphas_cumprod = abar(betas_linear)
t_list = [0, T // 10, T // 4, T // 2, 3 * T // 4, T - 1]
rows = []
for t in t_list:
    noise = torch.randn_like(x0)
    x_t = alphas_cumprod[t].sqrt() * x0 + (1 - alphas_cumprod[t]).sqrt() * noise   # q(x_t | x_0)
    rows.append(x_t.clamp(-1, 1))
show_images(torch.cat(rows), nrow=8, title="Forward process: rows are t = " + ", ".join(map(str, t_list)),
            save_path="outputs/02_forward_process.png", figsize=(8, 6))

### Build the conditional U-Net

In [ ]:
def build_unet():
    return ConditionalUNet(in_channels=1, base_channels=BASE_CH, emb_dim=128, num_classes=10)

unet = build_unet().to(device)
print(f"U-Net params: {sum(p.numel() for p in unet.parameters()):,}")
xt = torch.randn(4, 1, 28, 28, device=device); tt = torch.randint(0, T, (4,), device=device); yy = torch.arange(4, device=device)
out = unet(xt, tt, yy)
print("noise prediction shape:", tuple(out.shape)); assert out.shape == xt.shape

### Train (linear schedule)
Per batch: sample random `t`, noise the images with q-sampling, predict the noise, MSE loss. An EMA copy of the weights is returned and used for sampling.

In [ ]:
history_lin, ema_lin = train_diffusion(unet, train_loader, epochs=EPOCHS, device=device, timesteps=T,
                                       schedule="linear", lr=LR, checkpoint_dir="checkpoints",
                                       save_every=10, checkpoint_name="diffusion_linear")
plot_losses({"MSE loss": history_lin["loss"]}, "Diffusion training loss (linear schedule)", "outputs/02_diffusion_loss.png")

### Sample class-conditioned digits 0-9 (reverse diffusion)

In [ ]:
import time
labels = torch.arange(10).repeat_interleave(10)
t0 = time.time()
samples = sample_images(ema_lin, labels, betas_linear, device, num_inference_steps=SAMPLE_STEPS).cpu()
print(f"sampled {len(labels)} images in {time.time() - t0:.1f}s with {SAMPLE_STEPS} steps")
show_class_grid(samples, labels, per_class=10, title=f"Diffusion samples (linear schedule, {SAMPLE_STEPS} steps) - rows are labels 0-9",
                save_path="outputs/02_diffusion_class_grid.png")

### Watching the reverse process
Collect intermediate states by running the denoising loop for one image per class (full T steps).

In [ ]:
@torch.no_grad()
def reverse_trace(model, betas, labels, n_snap=8):
    """Same math as sample_images, but returns snapshots of x_t along the way (for visualisation)."""
    model.eval(); labels = labels.to(device); Tn = len(betas)
    b = betas.to(device); a = 1 - b; ab = torch.cumprod(a, 0)
    ab_prev = torch.cat([torch.ones(1, device=device), ab[:-1]])
    pv = b * (1 - ab_prev) / (1 - ab)
    x = torch.randn(len(labels), 1, 28, 28, device=device); snaps = []
    keep = set(np.linspace(Tn - 1, 0, n_snap).round().astype(int).tolist())
    for t in reversed(range(Tn)):
        eps = model(x, torch.full((len(labels),), t, device=device, dtype=torch.long), labels)
        x = (x - b[t] / (1 - ab[t]).sqrt() * eps) / a[t].sqrt()
        if t > 0: x = x + pv[t].sqrt() * torch.randn_like(x)
        if t in keep: snaps.append(x.clamp(-1, 1).cpu())
    return snaps

snaps = reverse_trace(ema_lin, betas_linear, torch.arange(10))
show_images(torch.cat([s[[3]] for s in snaps] + [s[[8]] for s in snaps]), nrow=len(snaps),
            title="Reverse process (left=noise, right=digit): digit 3 (top), digit 8 (bottom)",
            save_path="outputs/02_reverse_process.png", figsize=(10, 3))

### Stand-out: cosine noise schedule
Train an identical model with the cosine schedule; notebook 03 compares the FID of both.

In [ ]:
if TRAIN_COSINE:
    unet_cos = build_unet().to(device)
    history_cos, ema_cos = train_diffusion(unet_cos, train_loader, epochs=EPOCHS, device=device, timesteps=T,
                                           schedule="cosine", lr=LR, checkpoint_dir="checkpoints",
                                           save_every=10, checkpoint_name="diffusion_cosine")
    plot_losses({"linear": history_lin["loss"], "cosine": history_cos["loss"]}, "Training loss: linear vs cosine")
    s = sample_images(ema_cos, labels, betas_cosine, device, num_inference_steps=SAMPLE_STEPS).cpu()
    show_class_grid(s, labels, per_class=10, title="Diffusion samples (cosine schedule)", save_path="outputs/02_diffusion_cosine_grid.png")

### Notes
* Weights (raw + EMA) are stored in `checkpoints/diffusion_linear_final.pt` / `diffusion_cosine_final.pt` together with `T` and the schedule name.
* `sample_images(..., num_inference_steps=K)` re-spaces the chain so the *same* trained network can sample with fewer steps (K = T gives the full ancestral DDPM sampler). Notebook 03 uses this for the FID-vs-steps study.